# Retail Sales ETL & Data Quality Pipeline

**Beginner-built local ETL** · Synthetic 58-row retail input · Python / pandas / NumPy

This shareable notebook is reconstructed from my original learner-written VS Code code, preserving the step-by-step comments and decisions. The raw input is bundled beside this notebook, so no private `A:` drive path is needed. See [README.md](../README.md) for assumptions, scope and limitations.


In [ ]:
# Step 1: Import pandas, then load the raw CSV
import pandas as pd
from pathlib import Path

# Step 1.2: Set up logging before reading the source, so input failures are recorded
import logging
logging.basicConfig(filename="pipeline.log", level=logging.INFO, filemode="w", force=True)

# My source file was originally on drive A:; the GitHub version uses a portable relative file.
try:
    df = pd.read_csv(Path.cwd() / "retail_sales.csv")
    logging.info("Raw CSV loaded successfully")
except FileNotFoundError:
    logging.error("Raw CSV not found")
    raise


In [ ]:
# Step 2: Discover the data and investigate null values
# print(df.head(5))
# print(df.shape)
# print(df.isnull().sum())
# print(df[df["Quantity"].isnull()])      # Keep missing; do not invent sold quantity
# print(df[df["DiscountPct"].isnull()])   # Keep missing; unknown is not 0%
# print(df[df["ShippingState"].isnull()]) # Categorical unknown can be labelled

df_copy = df.copy()                    # Preserve original for comparison
df_copy = df_copy.fillna({"ShippingState": "Unknown"})

# Step 3: Identify and drop exact duplicate rows (keep first)
# print(df_copy.duplicated().sum())
df_copy = df_copy.drop_duplicates(keep="first").copy()
# print(df_copy.duplicated().sum())

# Step 4: Check data types
# print(df_copy.dtypes)

# Step 5: Correct types for analysis
# 5.1: Convert order dates
df_copy["OrderDate"] = pd.to_datetime(df_copy["OrderDate"], errors="coerce")
# 5.2: Convert prices safely: invalid text becomes missing, not an invented value
df_copy["UnitPrice"] = pd.to_numeric(df_copy["UnitPrice"], errors="coerce")
# 5.3: Quantity is a count but has missing values; use nullable Int64
df_copy["Quantity"] = df_copy["Quantity"].astype("Int64")


In [ ]:
# Step 6: Split rejected and accepted orders using the business-required columns
mask = df_copy["Quantity"].isnull() | df_copy["UnitPrice"].isnull()
rejected_orders = df_copy[mask].copy()
accepted = df_copy[~mask].copy()

# Step 6.3: Verify both groups
# print(rejected_orders.shape, accepted.shape)

# Step 6.4: Explain why each row is rejected
import numpy as np
condition = [
    rejected_orders["Quantity"].isnull(),
    rejected_orders["UnitPrice"].isnull(),
]
reason = ["Missing Quantity", "Invalid Unit Price"]
rejected_orders["Reject Reasons"] = np.select(condition, reason, default="others")

# Basic consistency checks before publishing outputs
assert len(accepted) + len(rejected_orders) == len(df_copy), "Some rows were lost or counted twice"


In [ ]:
# Step 7: Export processed files to both CSV and JSON
accepted.to_csv("retail_accepted_file.csv", index=False)
accepted.to_json("retail_accepted_file.json", orient="records", date_format="iso", index=False)
rejected_orders.to_csv("retail_rejected_file.csv", index=False)
rejected_orders.to_json("retail_rejected_file.json", orient="records", date_format="iso", index=False)

# Step 7.1: Verify saved CSVs by reading them back
accepted_file = pd.read_csv("retail_accepted_file.csv")
rejected_file = pd.read_csv("retail_rejected_file.csv")
assert len(accepted_file) == len(accepted)
assert len(rejected_file) == len(rejected_orders)

# Step 8: Log run results
logging.info(f"Raw Records: {len(df)}")
logging.info(f"Accepted Records: {len(accepted_file)}")
logging.warning(f"Rejected Records: {len(rejected_file)}")
print(f"Complete: {len(df)} raw, {len(df_copy)} unique, {len(accepted)} accepted, {len(rejected_orders)} rejected")
